#### Libraries for this entire Notebook

In [0]:
# (1) FOR PARAMETER READING FROM CONFIG FILES
import yaml

# (2) FOR AUTOLOADER => RUN_ID
import uuid

# (3) FOR AUTOLOADER => TIMESTAMP
from datetime import datetime, timezone
from pyspark.sql import functions as F


#### Logging of started notebook:


In [0]:
# NOTEBOOK NAME - Get notebook path using dbutils (works on serverless compute)
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

# Extract just the notebook name
notebook_name = notebook_path.split("/")[-1]

# NOTEBOOK START TIME - Current timestamp
notebook_start = datetime.now(timezone.utc).isoformat()

# LOGGING (kind-of): Notebook runtime environment - started notebook!
print(f"Notebook started with = notebook_path: {notebook_path}")
print(f"Notebook started with = notebook_name: {notebook_name}")
print(f"Notebook started with = notebook_start: {notebook_start}")

#### Project specific definitions for reusable PARAMETERS:


##### (0) Parameters: DROP DOWN selection

This fills the variable `referencedata_switch` to read the `CONFIG_<x>.YAML` files.

In [0]:
dbutils.widgets.dropdown(name= "referencedata_ui_dropdown", defaultValue= "weather_alert_reference", choices= ["asset_reference", "event_type_reference", "market_reference", "region_reference", "weather_alert_reference"], label="Reference Data File Selection")

referencedata_switch = dbutils.widgets.get("referencedata_ui_dropdown")

# LOGGING (kind-of): all => used Parameters for entire Notebook runtime environment!
print("Notebook`s Runtime Environment = DropDown value chosen for DATA PROCESSING:", referencedata_switch)

##### (1) Parameters: UNITY CATALOG objects

In [0]:
config_uc_objects__path = "/Workspace/Repos/michael.c.feiertag@web.de/PRJ__Vattenfall_EnergyOps_MarketIntelligence/config/config_uc_objects.yml"

with open(config_uc_objects__path, "r") as conf_uc:
    config_uc_objects = yaml.safe_load(conf_uc)

catalog_name = config_uc_objects["catalog"]
raw_schema = config_uc_objects["schemas"]["raw"]
landing_volume = config_uc_objects["volumes"]["landing"]
checkpoint_volume = config_uc_objects["volumes"]["checkpoints"]

# LOGGING (kind-of): all => used Parameters for entire Notebook runtime environment!
print(f"Used Config file: {config_uc_objects__path}")
print(f"Notebook`s Runtime Environment = catalog_name: {catalog_name}")
print(f"Notebook`s Runtime Environment = raw_schema: {raw_schema}")
print(f"Notebook`s Runtime Environment = landing_volume: {landing_volume}")
print(f"Notebook`s Runtime Environment = checkpoint_volume: {checkpoint_volume}")

##### (2) Parameters: UNITY CATALOG folders under volumes

In [0]:
config_uc_folders__path = "/Workspace/Repos/michael.c.feiertag@web.de/PRJ__Vattenfall_EnergyOps_MarketIntelligence/config/config_pathes.yml"

with open(config_uc_folders__path, "r") as conf_folders:
    config_uc_folders = yaml.safe_load(conf_folders)


# PROTOTYPING DROP-DOWN
#!referencedata_switch = "asset_reference"
#!referencedata_switch = "market_reference"


# SWITCH FOR REFERENCEDATA:---BEGIN:
if referencedata_switch == "asset_reference":

    raw_file_path__config = config_uc_folders["rawsource_paths"]["asset_reference"] 

    landing_folder__config = config_uc_folders["landing_paths"]["asset_reference"]
    checkpoint_folder__config = config_uc_folders["checkpoint_paths"]["asset_reference"]
    schema_tracking_folder__config = config_uc_folders["schema_tracking_paths"]["asset_reference"]

elif referencedata_switch == "event_type_reference":

    raw_file_path__config = config_uc_folders["rawsource_paths"]["event_type_reference"] 

    landing_folder__config = config_uc_folders["landing_paths"]["event_type_reference"]
    checkpoint_folder__config = config_uc_folders["checkpoint_paths"]["event_type_reference"]
    schema_tracking_folder__config = config_uc_folders["schema_tracking_paths"]["event_type_reference"]

elif referencedata_switch == "market_reference":

    raw_file_path__config = config_uc_folders["rawsource_paths"]["market_reference"] 

    landing_folder__config = config_uc_folders["landing_paths"]["market_reference"]
    checkpoint_folder__config = config_uc_folders["checkpoint_paths"]["market_reference"]
    schema_tracking_folder__config = config_uc_folders["schema_tracking_paths"]["market_reference"]

elif referencedata_switch == "region_reference":

    raw_file_path__config = config_uc_folders["rawsource_paths"]["region_reference"] 

    landing_folder__config = config_uc_folders["landing_paths"]["region_reference"]
    checkpoint_folder__config = config_uc_folders["checkpoint_paths"]["region_reference"]
    schema_tracking_folder__config = config_uc_folders["schema_tracking_paths"]["region_reference"]

elif referencedata_switch == "weather_alert_reference":
    raw_file_path__config = config_uc_folders["rawsource_paths"]["weather_alert_reference"] 

    landing_folder__config = config_uc_folders["landing_paths"]["weather_alert_reference"]
    checkpoint_folder__config = config_uc_folders["checkpoint_paths"]["weather_alert_reference"]
    schema_tracking_folder__config = config_uc_folders["schema_tracking_paths"]["weather_alert_reference"]

else:
    print("ERROR: Please check your config file for the correct pathes!")
# SWITCH FOR REFERENCEDATA:---END!

landing_path__config = f"/Volumes/{catalog_name}/{raw_schema}/{landing_volume}/{landing_folder__config}"
checkpoint_path__config = f"/Volumes/{catalog_name}/{raw_schema}/{checkpoint_volume}/{checkpoint_folder__config}"
schema_tracking_path__config = f"/Volumes/{catalog_name}/{raw_schema}/{checkpoint_volume}/{schema_tracking_folder__config}"

# LOGGING (kind-of): all => used Parameters for entire Notebook runtime environment!
print(f"Used Config file: {config_uc_folders__path}")
print(f"NBs Rt.Env.= raw_file_path__config: {raw_file_path__config}")
print(f"NBs Rt.Env.= landing_path__config: {landing_path__config}")
print(f"NBs Rt.Env.= checkpoint_path__config: {checkpoint_path__config}")
print(f"NBs Rt.Env.= schema_tracking_path__configs: {schema_tracking_path__config}")

##### (3) Parameters: UNITY CATALOG table under schema

In [0]:
config_uc_tables__path = "/Workspace/Repos/michael.c.feiertag@web.de/PRJ__Vattenfall_EnergyOps_MarketIntelligence/config/config_tables.yml"

with open(config_uc_tables__path, "r") as conf_tables:
    config_uc_tables = yaml.safe_load(conf_tables)


# SWITCH FOR REFERENCEDATA:---BEGIN:
if referencedata_switch == "asset_reference":
    bronze_table_name = config_uc_tables["bronze_tables"]["asset_reference"]
elif referencedata_switch == "event_type_reference":
    bronze_table_name = config_uc_tables["bronze_tables"]["event_type_reference"]
elif referencedata_switch == "market_reference":
    bronze_table_name = config_uc_tables["bronze_tables"]["market_reference"]
elif referencedata_switch == "region_reference":
    bronze_table_name = config_uc_tables["bronze_tables"]["region_reference"]
elif referencedata_switch == "weather_alert_reference":
    bronze_table_name = config_uc_tables["bronze_tables"]["weather_alert_reference"]
# SWITCH FOR REFERENCEDATA:---End!

# PARAMETER: ENTIRE PATH FOR TARGET TABLE NEEDED:
# - PARAMETERS (1) catalog_name, (2) raw_schema, BOTH determined in previous CELL.
# - PARAMETER (3) bronze_table_name determined above.
bronze_table = f"{catalog_name}.{raw_schema}.{bronze_table_name}"

# LOGGING (kind-of): all => used Parameters for entire Notebook runtime environment!
print(f"Used Config file: {config_uc_tables__path}")
print(f"Notebook`s Runtime Environment = bronze_table: {bronze_table}")

#### Copying RAW files from REPO to UNITY CATALOG folder

##### Parameters: SOURCE and TARGET pathes

In [0]:
# SOURCE: Parametrized from above cell
source_path = raw_file_path__config

# SOURCE: Parametrized from above set SWITCH 
# SWITCH FOR REFERENCEDATA:---BEGIN:
if referencedata_switch == "asset_reference":
    filename_pattern = "asset_reference"
elif referencedata_switch == "event_type_reference":
    filename_pattern = "event_type_reference"
elif referencedata_switch == "market_reference":
    filename_pattern = "market_reference"
elif referencedata_switch == "region_reference":
    filename_pattern = "region_reference"
elif referencedata_switch == "weather_alert_reference":
    filename_pattern = "weather_alert_reference"
# SWITCH FOR REFERENCEDATA:---End!

# TARGET: From above determination of Parameters:
landing_path = landing_path__config

# LOGGING (kind-of): all => used Parameters for next Cell!
print(f"Copy Files = source_path: {source_path}")
print(f"Copy Files = filename_pattern: {filename_pattern}")
print(f"Copy Files = landing_path: {landing_path}")

##### Copy files from SOURCE path to TARGET path

In [0]:
files = dbutils.fs.ls(source_path)

for file in files:
    if file.name.startswith(filename_pattern):
        dbutils.fs.cp(file.path, f"{landing_path}/{file.name}")
        print(f"Copy Files = File `{file.name}` copied to `{landing_path}/{file.name}`.")
    else:
        print(f"Copy Files = File `{file.name}` does not match the pattern to begin with `{filename_pattern}.`")

#### SIMPLE FULL LOADER
NOTES:

(1) A change from here for "DELTA AUTO-LOADER" solution is prepared in terms of Customizing in `CONFIG_PATHES.YAML` file.

(2) Also, that Customizing is already determined here in this Notebook in the cells above (see the Log/Prints for `checkpoint_path`and `schema_tracking_path`).

(3) Means for Change of Implementation for "DELTA AUTO-LOADER, refer to one of the `X_AUTOLOADER` notebooks (programs) in this Project REPO.

##### (1) SIMPLE FULL LOADER: DataFrame LOADER 

###### (1.1) SIMPLE FULL LOADER: DataFrame LOADER Source

In [0]:
# SOURCE = From above determination of Parameters - already used in above Cell "Copy Files":
load_from_path = landing_path

# LOGGING (kind-of): all => used Parameters for next Cell!
print(f"SIMPLE LOADER DF = Source: {load_from_path}")

###### (1.2) SIMPLE FULL LOADER: DataFrame LOADER enhancement column`s determination
####### PURPOSE:

This cell helps to define an enhanced DataFrame On-Top of the ingested schema from Source-Files. The PURPOSE is here ONLY for enhanced LOGGING of the Query Run and the ingested Source-Files in this Query Run. 

####### ENHANCEMENTS:

(1) ADD a new column for RUN-ID.

(2) ADD a new column for TIMESTAMP of ingestion.

(3) ADD a new column for FILENAME (with source path).

- The chosen funtionionality is: 
`.withColumn("ingest_source_file", F.col("_metadata.file_path")`".

- This solution respects the  CONSTRAINT here that the alternative function "`F.input-file-name`" is NOT ALLOWED in context of files managed by DATABRICKS UNITY CATALOG! 

In [0]:
# FOR ENHANCEMENT (1): DETERMINE A UNIQUE RUN-ID TO WRITE INTO BRONZE TABLE
ingest_run_id = str(uuid.uuid4())
print (f"Preparing to write with column 'ingest_run_id': {ingest_run_id},")

# FOR ENHANCEMENT (2): FREEZE CURRENT TIMESTAMP TO WRITE INTO BRONZE TABLE
ingest_run_ts = datetime.now(timezone.utc)
print (f"Preparing to write with column 'ingest_timestamp': {ingest_run_ts}.")

# FOR ENHANCEMENT (3): DETERMINE FILENAME TO WRITE INTO BRONZE TABLE
# - get filename directly from metadata during DataFrame LOADER enhancement column`s insertion
# = raw_referencedata_df.withColumn("ingest_source_file", F.col("_metadata.file_path"))


###### (1.3) SIMPLE FULL LOADER: DataFrame LOADER /1/ Schema Ingestion and /2/ Loading Data

In [0]:
raw_referencedata_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(load_from_path)
)

display(raw_referencedata_df)

###### (1.4) SIMPLE FULL LOADER: DataFrame LOADER enhancement column`s insertion

In [0]:
raw_referencedata_enh_df = (
    raw_referencedata_df
    .withColumn("ingest_run_id", F.lit(ingest_run_id))
    .withColumn("ingest_timestamp", F.lit(ingest_run_ts).cast("timestamp"))
    .withColumn("ingest_source_file", F.col("_metadata.file_path"))
)

display(raw_referencedata_enh_df)

##### (2) SIMPLE FULL LOADER: DataFrame WRITER 

###### (2.1) SIMPLE FULL LOADER: DataFrame WRITER Target

In [0]:
# TARGET = From above determination of Parameters - see above Cell "UNITY CATALOG table under schema":
bronze_table = bronze_table

# LOGGING (kind-of): all => used Parameters for next Cell!
print(f"SIMPLE LOADER DF = Target: {bronze_table}")

###### (2.2) SIMPLE FULL LOADER: DataFrame WRITER Execution

In [0]:
# TEMPORARY SETTING FOR RUN-STOP
# - "STOP" for controlled stop of entire notebook here
# - "GO" for regular run
switch_run = "GO"   #SET <= "STOP" / "GO"!

# BEG-OF "ACCORDING TO SETTING FOR RUN-STOP"
if switch_run == "STOP":
    raise Exception("STOP: Notebook execution stopped by user.")
# END-OF "ACCORDING TO SETTING FOR RUN-STOP"


# WRITE INTO BRONZE TABLE
(
    raw_referencedata_enh_df
    .write
    .mode("overwrite") # "append"(*) in case of delta
    .format("delta")
    .saveAsTable(bronze_table)
)
# **************************************
# (*)"append":
#     - good for bronze history
#     - more realistic for raw ingestion
#     - but reruns may duplicate data
# **************************************

# LOGGING (kind-of): VERIFICATION OF QUERY EXECUTION-STREAMING SUCCESS
print (f"Bronze ingestion finished writing into table `{bronze_table}`")
print (f"writing with column 'ingest_run_id': {ingest_run_id},")
print (f"writing with column 'ingest_timestamp': {ingest_run_ts}.")

##### (3) SIMPLE LOADER: success verification against TARGET


In [0]:
# TEMPORARY SETTING FOR DEBUGGING
# - "ON" for debugging variant
# - "OFF" for regular run
switch_debugging = "OFF"   #SET <= "ON" / "OFF"!

# BEG-OF "ACCORDING TO SETTING FOR DEBUGGING
if switch_debugging == "ON":
    bronze_table = "PRJ__vattenfall_01.raw.bronze_market_reference"
    ingest_run_ts = "2026-05-10 20:22:38.814865+00:00"
# END-OF "ACCORDING TO SETTING FOR DEBUGGING"

# LOGGING
print (f"Starting to read from table `{bronze_table}`")
print (f"reading with column `ingest_timestamp`: {ingest_run_ts}.")

# WORK WITH UNLIMITED DATAFRAME ON TOP OF BRONZE TABLE
bronze_df = spark.table(bronze_table)
bronze_total_count = bronze_df.count()

# OPTIMIZATION - Working with same UNLIMITED DATAFRAME
bronze_ingested_count = bronze_df.filter(F.col("ingest_timestamp") == F.lit(ingest_run_ts).cast("timestamp")).count()

print (f"Total records in bronze table: {bronze_total_count}")
print (f"Records ingested in bronze table: {bronze_ingested_count}")    

# BEG-OF "ACCORDING TO SETTING FOR DEBUGGING"
if switch_debugging == "ON":
    display(bronze_df)
else:
    display(bronze_df.filter(F.col("ingest_timestamp") == F.lit(ingest_run_ts).cast("timestamp")))
# END-OF "ACCORDING TO SETTING FOR DEBUGGING"

#### End-Of this Notebook